# PRAGMA Phase 5 - Padded reference model

Builds `PragmaConfig` from the fitted processor, instantiates `PragmaForMaskedModeling` (Profile State Encoder -> Event Encoder -> History Encoder -> `PragmaMLMHead`, all attention routed through `PaddedAttentionBackend` — section 7, ADR 0006/0009), and runs a real masked batch through forward + backward.

Checks the Phase 5 exit gate directly: tensor shapes, ~10M parameters at the paper's PRAGMA-S configuration, finite gradients everywhere, and the Event/History Encoder isolation property (an entity's embedding must not depend on which other entities share its batch).

In [1]:
from pathlib import Path

import torch

from pragma.config import MaskingConfig
from pragma.data import ParquetShardStore, PragmaCollator, TokenizedRecordDataset
from pragma.masking import MaskingPlanner
from pragma.modeling import PragmaConfig, PragmaForMaskedModeling
from pragma.processing import PragmaProcessor
from pragma.schema import SchemaRegistry

REPO_ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()

registry = SchemaRegistry.default()
processor = PragmaProcessor.load(REPO_ROOT / "data" / "processor", registry)
train_dataset = TokenizedRecordDataset.from_store(
    REPO_ROOT / "data" / "shards", ParquetShardStore(), split="train"
)
print(f"{len(train_dataset)} train records, vocab_size={processor.total_vocab_size}")

399 train records, vocab_size=412


## Build a batch with masking applied

Same `PragmaCollator` + `MaskingPlanner` pairing from `005_masking_inspection.ipynb`; the model consumes exactly the `PragmaBatch` contract Phase 3/4 established, unchanged.

In [2]:
planner = MaskingPlanner.from_registry(registry, processor.key_vocab, MaskingConfig())
collator = PragmaCollator(masking_planner=planner)
records = [train_dataset[i] for i in range(32) if len(train_dataset[i].events) > 0]
batch = collator(records)
batch.validate()
print(f"n_records={batch.n_records}, n_events={batch.n_events}, "
      f"n_event_tokens={batch.n_event_tokens}, n_masked={(batch.event_mlm_labels != -100).sum().item()}")

n_records=18, n_events=473, n_event_tokens=2452, n_masked=689


## Instantiate the model at the corpus's actual (small) vocab scale

`PragmaConfig.from_processor` reads `vocab_size`, `value_vocab_start`, and the `[USR]`/`[EVT]` special-token IDs straight from the fitted processor bundle (ADR 0009) — the model never touches the processor directly at runtime.

In [3]:
config = PragmaConfig.from_processor(
    processor,
    hidden_size=192,
    num_heads=3,
    intermediate_size=768,
    profile_layers=1,
    event_layers=5,
    history_layers=2,
)
model = PragmaForMaskedModeling(config)
n_params = sum(p.numel() for p in model.parameters())
print(f"vocab_size={config.vocab_size}")
print(f"n_params at this corpus's vocab: {n_params:,} ({n_params / 1e6:.2f}M)")

vocab_size=412
n_params at this corpus's vocab: 3,787,200 (3.79M)


## Parameter count at the paper's ~28,000-token vocabulary

Our synthetic corpus's vocabulary (a few hundred tokens) is far smaller than the paper's ~28k value tokens (section 6.1), so the embedding table — and therefore total parameter count — scales down accordingly. Confirming the architecture hits ~10M params **at the paper's vocab scale** is the actual exit-gate claim.

In [4]:
paper_scale_config = PragmaConfig(
    vocab_size=28060,
    value_vocab_start=64,
    hidden_size=192,
    num_heads=3,
    intermediate_size=768,
    profile_layers=1,
    event_layers=5,
    history_layers=2,
)
paper_scale_model = PragmaForMaskedModeling(paper_scale_config)
n_params_paper_scale = sum(p.numel() for p in paper_scale_model.parameters())
print(f"n_params at paper-scale vocab: {n_params_paper_scale:,} "
      f"({n_params_paper_scale / 1e6:.2f}M, target ~10M)")
assert 8_000_000 <= n_params_paper_scale <= 12_000_000

n_params at paper-scale vocab: 9,095,616 (9.10M, target ~10M)


## Forward + backward on the real batch

In [5]:
out = model(batch)
print(f"loss: {out.loss.item():.4f}")
print(f"mlm_logits shape: {tuple(out.mlm_logits.shape)}")
print(f"record_embeddings shape: {tuple(out.record_embeddings.shape)}")
print(f"event_embeddings shape: {tuple(out.event_embeddings.shape)}")
assert out.record_embeddings.shape == (batch.n_records, config.hidden_size)
assert out.event_embeddings.shape == (batch.n_events, config.hidden_size)

out.loss.backward()
n_no_grad = sum(1 for p in model.parameters() if p.grad is None)
n_nonfinite = sum(
    1 for p in model.parameters() if p.grad is not None and not torch.isfinite(p.grad).all()
)
print(f"params with no gradient: {n_no_grad} / {sum(1 for _ in model.parameters())}")
print(f"params with non-finite gradient: {n_nonfinite}")
assert n_no_grad == 0 and n_nonfinite == 0

loss: 5.9379
mlm_logits shape: (689, 385)
record_embeddings shape: (18, 192)
event_embeddings shape: (473, 192)


params with no gradient: 0 / 135
params with non-finite gradient: 0


## Isolation check: an event's embedding must not depend on its batch-mates

Same property `tests/unit/test_modeling.py` checks — run one record alone vs. inside a larger batch and confirm its event embeddings come out identical (up to floating-point noise from a different padding shape), proving the Event/History Encoders never let one customer's tokens leak into another's representation (ADR 0004).

In [6]:
model.eval()
target = next(r for r in train_dataset._records if len(r.events) >= 2)
others = [r for r in train_dataset._records if r.entity_id != target.entity_id and r.events][:6]

plain_collator = PragmaCollator()
solo_batch = plain_collator([target])
group_batch = plain_collator(others + [target])

with torch.no_grad():
    _, solo_event_emb, _ = model.pragma(solo_batch)
    _, group_event_emb, _ = model.pragma(group_batch)

n_target_events = len(target.events)
group_target_events = group_event_emb[-n_target_events:]
max_diff = (solo_event_emb - group_target_events).abs().max().item()
print(f"max abs difference (solo vs. inside a larger batch): {max_diff:.2e}")
assert torch.allclose(solo_event_emb, group_target_events, atol=1e-5)

max abs difference (solo vs. inside a larger batch): 5.36e-07
